# Exploración y decisiones de limpieza
Dataset OpenPowerlifting: 4.059.940 filas, cada una es un levantador en una competencia.

## 1. Carga
Se cargan 14 columnas.

In [1]:
import pandas as pd

columnas = ["Name", "Sex", "Event", "Equipment", "Age", "BodyweightKg",
            "Best3SquatKg", "Best3BenchKg", "Best3DeadliftKg",
            "TotalKg", "Dots", "Federation", "Date", "MeetCountry"]

df = pd.read_csv("../data/raw/openpowerlifting.csv",
                 usecols=columnas, low_memory=False)

df.shape

(4059940, 14)

## 2. Vistazo general
- Hay ~1 millón de nombres distintos: la misma persona aparece varias veces.
- EE.UU. y Rusia concentran cerca de dos tercios de las filas.

In [2]:
# primeras 5 filas
df.head()

,Name,Sex,Event,Equipment,Age,BodyweightKg,Best3SquatKg,Best3BenchKg,Best3DeadliftKg,TotalKg,Dots,Federation,Date,MeetCountry
0,E.S. Denisenko,F,B,Raw,28.5,67.3,NaN,NaN,NaN,NaN,NaN,GSF-Belarus,2019-06-22,Belarus
1,I.S. Lebetskaya,F,B,Raw,43.5,73.2,NaN,90.0,NaN,90.0,88.80,GSF-Belarus,2019-06-22,Belarus
2,K. Yakimovich,F,B,Raw,26.5,60.6,NaN,45.0,NaN,45.0,49.57,GSF-Belarus,2019-06-22,Belarus
3,A.G. Golneva,F,B,Raw,19.5,50.3,NaN,35.0,NaN,35.0,43.67,GSF-Belarus,2019-06-22,Belarus
4,E.V. Marunevskaya,F,B,Raw,19.5,63.7,NaN,42.5,NaN,42.5,45.40,GSF-Belarus,2019-06-22,Belarus


In [3]:
# tipos de datos y cantidad de nulos
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4059940 entries, 0 to 4059939
Data columns (total 14 columns):
 #   Column           Dtype  
---  ------           -----  
 0   Name             str    
 1   Sex              str    
 2   Event            str    
 3   Equipment        str    
 4   Age              float64
 5   BodyweightKg     float64
 6   Best3SquatKg     float64
 7   Best3BenchKg     float64
 8   Best3DeadliftKg  float64
 9   TotalKg          float64
 10  Dots             float64
 11  Federation       str    
 12  Date             str    
 13  MeetCountry      str    
dtypes: float64(7), str(7)
memory usage: 433.6 MB


In [4]:
# estadísticas básicas
df.describe()

,Age,BodyweightKg,Best3SquatKg,Best3BenchKg,Best3DeadliftKg,TotalKg,Dots
count,2.594094e+06,4.015475e+06,2.828954e+06,3.603146e+06,3.038877e+06,3.788676e+06,3.762437e+06
mean,3.069630e+01,8.377429e+01,1.710867e+02,1.146404e+02,1.867207e+02,3.799751e+02,2.799880e+02
std,1.334994e+01,2.268467e+01,6.702375e+01,5.250983e+01,6.218426e+01,2.009149e+02,1.258935e+02
min,3.500000e+00,1.500000e+01,-5.080200e+02,-5.225000e+02,-4.100000e+02,1.000000e+00,6.500000e-01
25%,2.050000e+01,6.686000e+01,1.200000e+02,7.250000e+01,1.375000e+02,2.109200e+02,1.622900e+02
50%,2.700000e+01,8.160000e+01,1.655600e+02,1.111300e+02,1.850000e+02,3.600000e+02,3.023200e+02
75%,3.800000e+01,9.830000e+01,2.150000e+02,1.475000e+02,2.313300e+02,5.325000e+02,3.738300e+02
max,1.055000e+02,3.000000e+02,5.950000e+02,6.590000e+02,4.925000e+02,1.407500e+03,8.180600e+02


In [5]:
# cantidad de registros por sexo
df["Sex"].value_counts()

Sex
M     2923658
F     1136082
Mx        200
Name: count, dtype: int64

In [6]:
# top 10 federaciones con mas registros
df["MeetCountry"].value_counts().head(10)

MeetCountry
USA          2174542
Russia        432052
Ukraine       112158
Canada         99547
Australia      94574
Germany        90524
England        85036
France         61170
Norway         60381
Finland        56140
Name: count, dtype: int64

## 3. Valores faltantes
Falta la edad en el 36% de las filas, la sentadilla en el 30%, el peso muerto en el 25% y la banca en el 11%.
Los nulos de sentadilla vienen de competencias que no la incluyen (100% nulos en B, D y BD).

In [7]:
# 1. Porcentaje de nulos por columna
(df.isna().mean() * 100).round(1)

Name                0.0
Sex                 0.0
Event               0.0
Equipment           0.0
Age                36.1
BodyweightKg        1.1
Best3SquatKg       30.3
Best3BenchKg       11.3
Best3DeadliftKg    25.1
TotalKg             6.7
Dots                7.3
Federation          0.0
Date                0.0
MeetCountry         0.0
dtype: float64

In [8]:
# 2. Cuántos nombres distintos hay (aproximación de levantadores únicos)
df["Name"].nunique()

1018414

In [9]:
# 3. Estadísticas de edad, peso corporal y total, para detectar valores raros
with pd.option_context("display.float_format", "{:,.1f}".format):
    display(df[["Age", "BodyweightKg", "TotalKg"]].describe())

,Age,BodyweightKg,TotalKg
count,"2,594,094.0","4,015,475.0","3,788,676.0"
mean,30.7,83.8,380.0
std,13.3,22.7,200.9
min,3.5,15.0,1.0
25%,20.5,66.9,210.9
50%,27.0,81.6,360.0
75%,38.0,98.3,532.5
max,105.5,300.0,"1,407.5"


In [10]:
# 4. Cuántos registros tiene Argentina
df[df["MeetCountry"] == "Argentina"].shape

(21788, 14)

# 4. Tipos de competencia
El 72% son SBD (sentadilla, banca y peso muerto). Solo en esas el total es comparable.

In [11]:
# Tipos de eventos: 
# S = Squat
# B = Benchpress
# D = Deadlift 
df["Event"].value_counts()

Event
SBD    2932314
B       804194
D       220458
BD       76241
S        20023
SB        4187
SD        2523
Name: count, dtype: int64

In [12]:
# Porcentaje de nulos en sentadilla según el tipo de competencia.
# Sirve para comprobar si los nulos vienen de competencias que no incluyen sentadilla.
df.groupby("Event")["Best3SquatKg"].apply(lambda s: s.isna().mean() * 100).round(1)

Event
B      100.0
BD     100.0
D      100.0
S        6.4
SB       9.0
SBD      4.4
SD       2.8
Name: Best3SquatKg, dtype: float64

## 5. Totales bajos
Fuera de SBD hay totales de 1 a 2,5 kg en adultos, probablemente errores de carga.
En SBD, la mayoría de los totales bajos son de niños.

In [13]:
df[df["TotalKg"] < 50] \
    .sort_values("TotalKg") \
    [["Name", "Event", "Age", "BodyweightKg", "TotalKg", "Federation"]] \
    .head(10)

,Name,Event,Age,BodyweightKg,TotalKg,Federation
2694375,Vadim Tokarev,B,54.5,83.0,1.0,FPR
3140216,Veiko Friedrich,B,NaN,80.5,1.0,BVDK
3140196,Sören Radtke,B,21.0,89.3,1.0,BVDK
3140233,Rainer Geese,B,NaN,65.7,1.0,BVDK
3140192,Alex Wambolt,B,22.0,78.5,2.0,BVDK
3140237,Jürgen Groß,B,63.5,78.8,2.0,BVDK
3140183,Chris Konzagk,B,NaN,89.7,2.0,BVDK
2297077,T. McGuire,B,NaN,110.0,2.5,WDFPF
1005861,Raquel Kavilhuka,D,43.5,68.0,2.5,CBLB
1005908,Valdinei Barbosa,B,34.5,81.0,2.5,CBLB


In [14]:
df[df["Event"] == "SBD"] \
    .nsmallest(30, "TotalKg") \
    [["Name", "Event", "Age","TotalKg", "BodyweightKg", "Federation"]]

,Name,Event,Age,TotalKg,BodyweightKg,Federation
3223010,Elliott Peterson,SBD,9.0,7.00,41.60,RPS
3240459,Aiden Giarolo,SBD,8.0,7.00,35.70,RPS
3240474,Aiden Giarolo,SBD,8.0,7.00,35.70,RPS
3240361,Aubree Miller,SBD,9.0,8.00,40.00,RPS
346448,Nikol Malykh,SBD,6.0,24.00,27.20,WRPF
2498548,J. Waltz,SBD,NaN,25.00,48.00,USA-UA
630134,Charles Mosher,SBD,5.0,28.60,22.50,RAW-CAN
3196527,Aria Green,SBD,6.0,34.00,44.00,RPS
2285385,Marcus Murphy,SBD,NaN,34.02,56.00,USPF
3216544,Kinley Perry,SBD,NaN,34.02,19.00,RPS


## 6. Duplicados
108.073 filas son duplicados exactos, probablemente la misma actuación cargada dos veces. Se eliminan.

In [15]:
sbd = df[df["Event"] == "SBD"].dropna(subset=["TotalKg", "BodyweightKg", "Dots"])
print(sbd.duplicated().sum())   # da 108073

dups = sbd[sbd.duplicated(keep=False)].sort_values(["Name", "Date"])
dups[["Name", "Date", "Federation", "BodyweightKg", "TotalKg"]].head(10)
dups["Federation"].value_counts().head(10)

108073


Federation
USAPL         51333
USPA          46185
USPF           8738
NASA           6260
APF            5880
RPS            4967
RAW            4391
EPA            4268
WRPF           4216
WPC-Poland     3400
Name: count, dtype: int64

## Decisiones
Solo SBD, sin nulos en total, peso corporal y Dots, sin duplicados: quedan 2.591.805 filas.